class ActorRolloutRefWorker(MegatronWorker, DistProfilerExtension):
- \_\_init\_\_()
- _build_model_optimizer()
- _build_rollout()
- init_model()
- <font color='red'>update_actor()</font>
- generate_sequences()
- compute_ref_log_prob()
- compute_log_prob()
- load_checkpoint()
- load_pretrained_model()
- save_checkpoint()

In [ ]:
    @register(dispatch_mode=make_nd_compute_dataproto_dispatch_fn(mesh_name="actor"))
    @GPUMemoryLogger(role="update_actor", logger=logger)
    @DistProfiler.annotate(color="red")
    def update_actor(self, data: DataProto):
        assert self._is_actor
        if self._is_offload_param:
            load_megatron_model_to_gpu(self.actor_module)
            log_gpu_memory_usage("After load actor params and grad during update_actor", logger=logger)
        if self._is_offload_optimizer:
            load_megatron_optimizer(self.actor_optimizer)
            log_gpu_memory_usage("After load actor optimizer during update_actor", logger=logger)
        data.batch = data.batch.to(get_device_name())

        micro_batch_size = self.config.actor.ppo_micro_batch_size_per_gpu
        data.meta_info["micro_batch_size"] = micro_batch_size
        dataloader = self.actor.make_minibatch_iterator(data=data)
        with Timer(name="update_policy", logger=None) as timer:
            metrics = self.actor.update_policy(dataloader=dataloader)
        delta_time = timer.last
        global_num_tokens = data.meta_info["global_token_num"]
        estimated_flops, promised_flops = self.flops_counter.estimate_flops(global_num_tokens, delta_time)
        metrics["perf/mfu/actor"] = estimated_flops * self.config.actor.ppo_epochs / promised_flops / self.world_size
        metrics["perf/max_memory_allocated_gb"] = get_torch_device().max_memory_allocated() / (1024**3)
        metrics["perf/max_memory_reserved_gb"] = get_torch_device().max_memory_reserved() / (1024**3)
        metrics["perf/cpu_memory_used_gb"] = psutil.virtual_memory().used / (1024**3)
        from verl.utils.megatron.optimizer import get_megatron_last_lr

        metrics["actor/lr"] = get_megatron_last_lr(self.actor_optimizer)
        self.actor_optimizer_scheduler.step(1)

        # TODO: here, we should return all metrics
        output = DataProto(meta_info={"metrics": metrics})
        output = output.to("cpu")

        if self._is_offload_param:
            offload_megatron_model_to_cpu(self.actor_module)
            log_gpu_memory_usage("After offload actor params and grad during update_actor", logger=logger)
        if self._is_offload_optimizer:
            offload_megatron_optimizer(self.actor_optimizer)
            log_gpu_memory_usage("After offload actor optimizer during update_actor", logger=logger)

        aggressive_empty_cache(force_sync=True)
        return output


# update_actor
- 这段代码是一个基于 Megatron-LM 框架、针对 Actor 模型（策略网络） 进行分布式训练和参数更新的核心函数。
- 从装饰器 make_nd_compute_dataproto_dispatch_fn 和 PPO 相关的配置来看，它主要用于大规模语言模型（LLM）的强化学习（如 PPO 算法）训练阶段。

为了让你更清晰地理解，我将这段代码的执行流程拆解为 准备阶段、核心计算阶段 和 收尾阶段 三个部分进行详细介绍：
## 🚀 一、准备阶段：环境与数据就绪
### 前置检查与装饰器
- 函数被三个装饰器包裹：@register 负责分布式任务的分发调度；@GPUMemoryLogger 用于在函数执行前后自动记录显存使用情况；@DistProfiler 用于分布式性能剖析（标记为红色，便于在性能分析工具中识别）。
assert self._is_actor：确保当前运行的进程确实是负责 Actor 任务的进程。
### 参数与优化器加载（Offload 机制）
- 代码中引入了 Offload（卸载/交换） 机制。当显存不足以同时放下模型参数和优化器状态时，会将它们暂时存放在 CPU 内存中。
- if self._is_offload_param:：如果开启了参数卸载，调用 load_megatron_model_to_gpu 将 Actor 模型的参数和梯度从 CPU 搬运回 GPU。
- if self._is_offload_optimizer:：如果开启了优化器卸载，调用 load_megatron_optimizer 将优化器状态加载到 GPU。
- 每次搬运后都会调用 log_gpu_memory_usage 记录当前显存状态，方便排查显存溢出（OOM）问题。
### 数据准备
- data.batch = data.batch.to(get_device_name())：将输入的训练数据（DataProto）搬运到当前 GPU 设备上。
- 设置 micro_batch_size（微批次大小），并调用 make_minibatch_iterator 将数据切分成多个小批次（minibatch），为接下来的多次迭代（PPO epochs）做准备。
## ⚙️ 二、核心计算阶段：策略更新与指标统计
### 执行策略更新
- with Timer(...)：开启一个计时器，精确记录模型更新所消耗的时间（delta_time）。
- metrics = self.actor.update_policy(dataloader=dataloader)：这是整个函数的核心。它调用底层 Actor 模型的 update_policy 方法，传入准备好的 dataloader，执行前向传播、计算 PPO 裁剪损失、反向传播以及梯度更新。同时返回包含各项损失（Loss）的指标字典。
### 性能指标（Perf Metrics）计算
- MFU（Model FLOPs Utilization，模型算力利用率）：通过 self.flops_counter.estimate_flops 结合训练消耗的 tokens 数量和时间，估算实际达到的算力，再除以硬件理论峰值算力（promised_flops），得到 MFU。这是衡量大规模模型训练效率的关键指标。
- 显存与内存监控：记录 GPU 的最大分配显存（max_memory_allocated_gb）、最大预留显存（max_memory_reserved_gb）以及当前 CPU 的内存使用量。
- 学习率（LR）记录：通过 get_megatron_last_lr 获取 Megatron 优化器当前的实际学习率并记录。
### 更新学习率调度器
- self.actor_optimizer_scheduler.step(1)：执行一步学习率调度，通常用于按照预设的策略（如余弦退火）衰减学习率。

## 🏁 三、收尾阶段：资源释放与返回
1. 封装输出
- 将计算出的各项 metrics 封装回一个 DataProto 对象中，并调用 .to("cpu") 将其转移回 CPU，以便主进程收集日志或进行后续处理。
2. 参数与优化器卸载（Offload 机制）
- 与准备阶段相反，如果开启了 Offload，此时需要将 GPU 上的资源释放回 CPU，以节省宝贵的显存空间供其他模块（如 Critic 或 Reference 模型）使用。
- 分别调用 offload_megatron_model_to_cpu 和 offload_megatron_optimizer 完成卸载，并再次记录显存日志。
3. 强制清理缓存
- aggressive_empty_cache(force_sync=True)：强制清理 PyTorch 的显存缓存碎片，进一步防止显存泄漏或碎片化导致的 OOM 问题。

总结来说，这段代码是一个高度工程化、针对大规模分布式训练优化的 Actor 更新接口。它不仅完成了核心的梯度更新任务，还深度集成了显存优化（Offload）、性能监控（MFU、显存用量）以及分布式调度等功能，是保障大模型 RL 训练稳定、高效运行的关键环节。

